# Lab 27 — Fixed Hungarian holdout with nested optimization

Colab 27 phát triển từ protocol của Colab 26:

    Test cố định: Hungarian
    Development: Cleveland, Switzerland, VA
    Inner CV: 2 train + 1 validation, xoay theo hospital
    Final fit: cả 3 development hospitals
    Final test: Hungarian đúng một lần

Các nhánh thí nghiệm:

- F0: baseline cố định của Colab 26 — Logistic Regression và LightGBM.
- F1: Optuna nested cho Logistic Regression, CatBoost, LightGBM và MLP.
- F2: Optuna + SMOTENC train-only cho LR, LightGBM và MLP.
- F3: bootstrap-jitter synthetic augmentation train-only, tắt mặc định vì cần biện minh y khoa.
- Stacking: nested OOF và chỉ bật nếu đồng thời cải thiện worst-site ROC-AUC và Recall tối thiểu 0.005.

Nguyên tắc leakage:

- Preprocessing, Optuna, SMOTENC và synthetic augmentation chỉ được fit/tạo trong training split.
- Hungarian không dùng để chọn hyperparameter, threshold, seed hay bật stacking.
- Kết quả Hungarian của Colab 24/25 đã từng được xem, nên kết quả ở Colab 27 phải ghi là exploratory fixed-holdout evaluation, không phải blind tuyệt đối.
- Không sử dụng Accuracy làm metric chính; ưu tiên ROC-AUC, PR-AUC, Recall, Brier và worst-site.

> Đây là nghiên cứu trên dữ liệu công khai, không phải bằng chứng lâm sàng hay công cụ chẩn đoán.

In [ ]:
!pip -q install optuna lightgbm catboost imbalanced-learn seaborn

In [ ]:
import json
import random
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import tensorflow as tf
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTENC
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option('display.max_columns', 100)
tf.get_logger().setLevel('ERROR')

RANDOM_STATE = 42
FINAL_SEED = 42
INNER_SEEDS = (42, 123, 2025)
THRESHOLD = 0.50
N_TRIALS = 20
STACKING_OOF_TRIALS = 8
STACKING_MIN_IMPROVEMENT = 0.005
MAX_EPOCHS = 150
PATIENCE = 15

RUN_BASELINE = True
RUN_OPTUNA = True
RUN_SMOTENC = True
RUN_SYNTHETIC = False
RUN_STACKING = True

BASELINE_MODEL_NAMES = ['Logistic Regression', 'LightGBM']
MODEL_NAMES = ['Logistic Regression', 'CatBoost', 'LightGBM', 'MLP']

OUTPUT_DIR = Path('/content/uci_multicenter_fixed_hungarian_advanced_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal'
]
TARGET = 'target'
NUMERICAL_FEATURES = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
CATEGORICAL_FEATURES = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

BASE_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease'
FILES = {
    'cleveland': 'processed.cleveland.data',
    'hungarian': 'processed.hungarian.data',
    'switzerland': 'processed.switzerland.data',
    'va': 'processed.va.data',
}
EXPECTED_ROWS = {'cleveland': 303, 'hungarian': 294, 'switzerland': 123, 'va': 200}
DEVELOPMENT_SITES = ['cleveland', 'switzerland', 'va']
FINAL_TEST_SITE = 'hungarian'
COLUMNS = FEATURES + ['num']

LOCAL_DATA_DIR_CANDIDATES = [
    Path('/content/heart-disease-diagnosis/data/raw/uci_multicenter'),
    Path('/content/data/raw/uci_multicenter'),
    Path('data/raw/uci_multicenter'),
    Path('../data/raw/uci_multicenter'),
]
LOCAL_DATA_DIR = next(
    (path for path in LOCAL_DATA_DIR_CANDIDATES if path.exists()),
    None,
)

print('Development sites:', DEVELOPMENT_SITES)
print('Final test site:', FINAL_TEST_SITE)
print('Models:', MODEL_NAMES)
print('Optuna trials:', N_TRIALS)
print('SMOTENC:', RUN_SMOTENC, '| synthetic:', RUN_SYNTHETIC, '| stacking:', RUN_STACKING)

In [ ]:
def read_uci(site, filename):
    source = (LOCAL_DATA_DIR / filename) if LOCAL_DATA_DIR else f'{BASE_URL}/{filename}'
    frame = pd.read_csv(
        source,
        names=COLUMNS,
        na_values=['?'],
        skipinitialspace=True,
    )
    frame = frame.apply(pd.to_numeric, errors='coerce')
    frame[TARGET] = (frame['num'] > 0).astype('int8')
    frame['site'] = site
    return frame[FEATURES + [TARGET, 'site']]


data = pd.concat(
    [read_uci(site, filename) for site, filename in FILES.items()],
    ignore_index=True,
)
assert len(data) == 920

development = data[data['site'].isin(DEVELOPMENT_SITES)].reset_index(drop=True)
final_test = data[data['site'] == FINAL_TEST_SITE].reset_index(drop=True)

assert len(development) == 626
assert len(final_test) == 294
assert set(development['site']) == set(DEVELOPMENT_SITES)
assert not set(development['site']).intersection({FINAL_TEST_SITE})

for site, expected_rows in EXPECTED_ROWS.items():
    assert int((data['site'] == site).sum()) == expected_rows

development_summary = development.groupby('site').agg(
    rows=(TARGET, 'size'),
    disease_count=(TARGET, 'sum'),
    positive_rate=(TARGET, 'mean'),
).reset_index()
development_missing = development.groupby('site')[FEATURES].apply(
    lambda frame: frame.isna().mean()
).T

print('Data source:', str(LOCAL_DATA_DIR) if LOCAL_DATA_DIR else 'UCI URL fallback')
print('Development shape:', development.shape)
print('Reserved final-test rows:', len(final_test), '(labels not used in development)')
display(development_summary.round(4))
display(pd.crosstab(development['site'], development[TARGET], margins=True))
display((development_missing * 100).round(1))

development_summary.to_csv(OUTPUT_DIR / 'development_site_summary.csv', index=False)
development_missing.to_csv(OUTPUT_DIR / 'development_missing_by_site.csv')

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


def apply_p1(frame):
    out = frame.copy()
    for column in FEATURES:
        out[column] = pd.to_numeric(out[column], errors='coerce')
    for column in ['trestbps', 'chol']:
        out.loc[out[column] <= 0, column] = np.nan
    return out


def prepare_dense(frame):
    ready = apply_p1(frame)
    return ready[FEATURES], ready[TARGET].to_numpy()


def prepare_catboost(frame):
    ready = apply_p1(frame)
    features = ready[FEATURES].copy()
    for column in FEATURES:
        features[f'{column}__missing'] = features[column].isna().astype('int8')
    for column in CATEGORICAL_FEATURES:
        features[column] = features[column].fillna('__MISSING__').astype(str)
    return features, ready[TARGET].to_numpy()


def make_preprocessor(scale_numeric=True):
    numerical_steps = [
        ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ]
    if scale_numeric:
        numerical_steps.append(('scaler', StandardScaler()))

    return ColumnTransformer([
        ('numerical', Pipeline(numerical_steps), NUMERICAL_FEATURES),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent', add_indicator=True)),
            ('encoder', OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False,
            )),
        ]), CATEGORICAL_FEATURES),
    ])


def make_preprocessor_compat(scale_numeric=True):
    try:
        return make_preprocessor(scale_numeric)
    except TypeError:
        numerical_steps = [
            ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
        ]
        if scale_numeric:
            numerical_steps.append(('scaler', StandardScaler()))
        return ColumnTransformer([
            ('numerical', Pipeline(numerical_steps), NUMERICAL_FEATURES),
            ('categorical', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent', add_indicator=True)),
                ('encoder', OneHotEncoder(handle_unknown='ignore', sparse=False)),
            ]), CATEGORICAL_FEATURES),
        ])


def balanced_positive_weight(y):
    negative_count = max(1, int((y == 0).sum()))
    positive_count = max(1, int((y == 1).sum()))
    return negative_count / positive_count


def add_train_only_synthetic(train_frame, seed, multiplier=0.5):
    # Exploratory bootstrap-jitter only. Created from this training split.
    # Disabled by default because medical plausibility needs review.
    ready = apply_p1(train_frame)
    rng = np.random.default_rng(seed)
    n_synthetic = max(1, int(len(ready) * multiplier))
    synthetic = ready.sample(
        n=n_synthetic,
        replace=True,
        random_state=seed,
    ).reset_index(drop=True)

    for column in NUMERICAL_FEATURES:
        observed = ready[column].dropna()
        scale = float(observed.std()) if len(observed) > 1 else 1.0
        jitter = rng.normal(0.0, max(scale * 0.02, 1e-3), size=n_synthetic)
        values = synthetic[column].to_numpy(dtype='float64')
        values = values + jitter
        if len(observed):
            values = np.clip(values, float(observed.min()), float(observed.max()))
        synthetic[column] = values

    synthetic['site'] = 'train_only_synthetic'
    return pd.concat([
        ready[FEATURES + [TARGET, 'site']],
        synthetic[FEATURES + [TARGET, 'site']],
    ], ignore_index=True)


def apply_train_sampling(train_frame, sampling, seed):
    if sampling == 'none':
        return train_frame.reset_index(drop=True)
    if sampling == 'bootstrap_jitter':
        return add_train_only_synthetic(train_frame, seed)
    if sampling != 'smotenc':
        raise ValueError(f'Unknown sampling mode: {sampling}')

    ready = apply_p1(train_frame)
    y = ready[TARGET].to_numpy()
    if np.bincount(y).min() < 2:
        return ready.reset_index(drop=True)

    numerical_imputer = SimpleImputer(strategy='median')
    categorical_imputer = SimpleImputer(strategy='most_frequent')
    X_numerical = numerical_imputer.fit_transform(ready[NUMERICAL_FEATURES])
    X_categorical = categorical_imputer.fit_transform(ready[CATEGORICAL_FEATURES])
    X_for_smote = np.column_stack([X_numerical, X_categorical])

    categorical_indices = list(range(
        len(NUMERICAL_FEATURES),
        len(NUMERICAL_FEATURES) + len(CATEGORICAL_FEATURES),
    ))
    minimum_class_count = int(np.bincount(y).min())
    k_neighbors = max(1, min(5, minimum_class_count - 1))
    sampler = SMOTENC(
        categorical_features=categorical_indices,
        k_neighbors=k_neighbors,
        random_state=seed,
    )
    X_resampled, y_resampled = sampler.fit_resample(X_for_smote, y)

    smote_columns = NUMERICAL_FEATURES + CATEGORICAL_FEATURES
    resampled = pd.DataFrame(X_resampled, columns=smote_columns)
    for column in CATEGORICAL_FEATURES:
        resampled[column] = np.rint(resampled[column]).astype(float)
    resampled[TARGET] = y_resampled.astype('int8')
    resampled['site'] = 'train_only_smotenc'
    return resampled[FEATURES + [TARGET, 'site']]


FIXED_PARAMS = {
    'Logistic Regression': {
        'C': 1.0,
        'solver': 'lbfgs',
        'max_iter': 2000,
    },
    'CatBoost': {
        'iterations': 300,
        'depth': 5,
        'learning_rate': 0.03,
        'l2_leaf_reg': 3.0,
        'random_strength': 1.0,
        'border_count': 64,
    },
    'LightGBM': {
        'n_estimators': 250,
        'learning_rate': 0.03,
        'num_leaves': 15,
        'min_child_samples': 15,
        'subsample': 0.9,
        'colsample_bytree': 0.9,
        'reg_lambda': 1.0,
    },
    'MLP': {
        'hidden1': 32,
        'hidden2': 16,
        'dropout': 0.20,
        'learning_rate': 0.001,
        'l2_reg': 1e-4,
        'batch_size': 32,
        'max_epochs': MAX_EPOCHS,
        'patience': PATIENCE,
    },
}

In [ ]:
def suggest_params(trial, model_name):
    if model_name == 'Logistic Regression':
        return {
            'C': trial.suggest_float('C', 0.01, 10.0, log=True),
            'solver': trial.suggest_categorical('solver', ['lbfgs', 'liblinear']),
            'max_iter': 2000,
            'positive_weight': trial.suggest_categorical(
                'positive_weight',
                [0.75, 1.0, 1.25, 1.5, 2.0],
            ),
        }
    if model_name == 'CatBoost':
        return {
            'iterations': trial.suggest_int('iterations', 150, 600),
            'depth': trial.suggest_int('depth', 3, 7),
            'learning_rate': trial.suggest_float(
                'learning_rate', 0.01, 0.20, log=True
            ),
            'l2_leaf_reg': trial.suggest_float(
                'l2_leaf_reg', 1e-2, 20.0, log=True
            ),
            'random_strength': trial.suggest_float(
                'random_strength', 0.0, 3.0
            ),
            'border_count': trial.suggest_categorical(
                'border_count',
                [32, 64, 128],
            ),
            'positive_weight': trial.suggest_categorical(
                'positive_weight',
                [0.75, 1.0, 1.25, 1.5, 2.0],
            ),
        }
    if model_name == 'LightGBM':
        return {
            'n_estimators': trial.suggest_int('n_estimators', 100, 500),
            'learning_rate': trial.suggest_float(
                'learning_rate', 0.01, 0.20, log=True
            ),
            'num_leaves': trial.suggest_int('num_leaves', 7, 63),
            'min_child_samples': trial.suggest_int(
                'min_child_samples', 10, 50
            ),
            'subsample': trial.suggest_float('subsample', 0.70, 1.00),
            'colsample_bytree': trial.suggest_float(
                'colsample_bytree', 0.70, 1.00
            ),
            'reg_lambda': trial.suggest_float(
                'reg_lambda', 1e-3, 10.0, log=True
            ),
            'positive_weight': trial.suggest_categorical(
                'positive_weight',
                [0.75, 1.0, 1.25, 1.5, 2.0],
            ),
        }
    if model_name == 'MLP':
        return {
            'hidden1': trial.suggest_categorical('hidden1', [16, 32, 64]),
            'hidden2': trial.suggest_categorical('hidden2', [8, 16, 32]),
            'dropout': trial.suggest_float('dropout', 0.0, 0.40),
            'learning_rate': trial.suggest_float(
                'learning_rate', 1e-4, 1e-2, log=True
            ),
            'l2_reg': trial.suggest_float(
                'l2_reg', 1e-6, 1e-2, log=True
            ),
            'batch_size': trial.suggest_categorical(
                'batch_size',
                [16, 32, 64],
            ),
            'max_epochs': MAX_EPOCHS,
            'patience': PATIENCE,
        }
    raise ValueError(f'Unknown model: {model_name}')


def build_estimator(model_name, params, seed, y_train):
    model_params = dict(params)
    positive_weight = model_params.pop('positive_weight', None)

    if model_name == 'Logistic Regression':
        class_weight = (
            'balanced'
            if positive_weight is None
            else {0: 1.0, 1: float(positive_weight)}
        )
        return LogisticRegression(
            class_weight=class_weight,
            random_state=seed,
            **model_params,
        )

    if model_name == 'LightGBM':
        class_weight = (
            'balanced'
            if positive_weight is None
            else {0: 1.0, 1: float(positive_weight)}
        )
        return LGBMClassifier(
            class_weight=class_weight,
            random_state=seed,
            verbosity=-1,
            n_jobs=1,
            **model_params,
        )

    if model_name == 'CatBoost':
        scale_pos_weight = (
            balanced_positive_weight(y_train)
            if positive_weight is None
            else float(positive_weight)
        )
        return CatBoostClassifier(
            loss_function='Logloss',
            eval_metric='AUC',
            random_seed=seed,
            scale_pos_weight=scale_pos_weight,
            verbose=False,
            allow_writing_files=False,
            thread_count=2,
            **model_params,
        )

    raise ValueError(f'Unknown estimator: {model_name}')


def build_mlp(input_dim, params, seed):
    seed_everything(seed)
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(
            int(params['hidden1']),
            activation='relu',
            kernel_regularizer=l2(float(params['l2_reg'])),
        ),
        Dropout(float(params['dropout'])),
        Dense(
            int(params['hidden2']),
            activation='relu',
            kernel_regularizer=l2(float(params['l2_reg'])),
        ),
        Dropout(float(params['dropout'])),
        Dense(1, activation='sigmoid'),
    ])
    model.compile(
        optimizer=Adam(learning_rate=float(params['learning_rate'])),
        loss='binary_crossentropy',
        metrics=[tf.keras.metrics.AUC(name='auc')],
    )
    return model


def fit_mlp(X_train, y_train, X_valid, y_valid, params, seed):
    tf.keras.backend.clear_session()
    model = build_mlp(X_train.shape[1], params, seed)
    reduce_lr = ReduceLROnPlateau(
        monitor='val_auc',
        mode='max',
        factor=0.5,
        patience=max(3, int(params['patience']) // 3),
        min_lr=1e-6,
        verbose=0,
    )
    early_stop = EarlyStopping(
        monitor='val_auc',
        mode='max',
        patience=int(params['patience']),
        min_delta=1e-4,
        restore_best_weights=True,
        verbose=0,
    )
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_valid, y_valid),
        epochs=int(params['max_epochs']),
        batch_size=int(params['batch_size']),
        callbacks=[reduce_lr, early_stop],
        class_weight={
            0: 1.0,
            1: balanced_positive_weight(y_train),
        },
        verbose=0,
        shuffle=True,
    )
    values = history.history.get('val_auc', [])
    best_epoch = int(np.argmax(values) + 1) if values else len(
        history.history['loss']
    )
    return model, best_epoch


def score_probability(y_true, probability):
    prediction = (probability >= THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        prediction,
        labels=[0, 1],
    ).ravel()
    return {
        'accuracy': accuracy_score(y_true, prediction),
        'precision': precision_score(y_true, prediction, zero_division=0),
        'recall': recall_score(y_true, prediction, zero_division=0),
        'pr_auc': average_precision_score(y_true, probability),
        'specificity': tn / (tn + fp) if (tn + fp) else np.nan,
        'f1': f1_score(y_true, prediction, zero_division=0),
        'roc_auc': (
            roc_auc_score(y_true, probability)
            if len(np.unique(y_true)) > 1 else np.nan
        ),
        'brier': brier_score_loss(y_true, probability),
        'false_negatives': int(fn),
        'false_positives': int(fp),
    }

In [ ]:
def fit_predict_base(
    model_name,
    train_frame,
    predict_frame,
    params,
    seed,
    sampling='none',
    validation_frame=None,
):
    params = dict(params)
    fit_frame = apply_train_sampling(train_frame, sampling, seed)

    if model_name in ['Logistic Regression', 'LightGBM']:
        X_train, y_train = prepare_dense(fit_frame)
        X_predict, _ = prepare_dense(predict_frame)
        preprocessor = make_preprocessor_compat(
            scale_numeric=(model_name == 'Logistic Regression')
        )
        X_train = preprocessor.fit_transform(X_train).astype('float32')
        X_predict = preprocessor.transform(X_predict).astype('float32')
        estimator = build_estimator(model_name, params, seed, y_train)
        estimator.fit(X_train, y_train)
        return estimator.predict_proba(X_predict)[:, 1]

    if model_name == 'CatBoost':
        X_train, y_train = prepare_catboost(fit_frame)
        X_predict, _ = prepare_catboost(predict_frame)
        estimator = build_estimator(model_name, params, seed, y_train)
        estimator.fit(X_train, y_train, cat_features=CATEGORICAL_FEATURES)
        return estimator.predict_proba(X_predict)[:, 1]

    if model_name == 'MLP':
        X_train, y_train = prepare_dense(fit_frame)
        X_predict, _ = prepare_dense(predict_frame)
        preprocessor = make_preprocessor_compat(scale_numeric=True)
        X_train = preprocessor.fit_transform(X_train).astype('float32')
        X_predict = preprocessor.transform(X_predict).astype('float32')

        if validation_frame is not None:
            X_valid, y_valid = prepare_dense(validation_frame)
            X_valid = preprocessor.transform(X_valid).astype('float32')
            model, _ = fit_mlp(
                X_train,
                y_train,
                X_valid,
                y_valid,
                params,
                seed,
            )
        else:
            epochs = int(params.get('fit_epochs', params['max_epochs']))
            tf.keras.backend.clear_session()
            model = build_mlp(X_train.shape[1], params, seed)
            model.fit(
                X_train,
                y_train,
                epochs=epochs,
                batch_size=int(params['batch_size']),
                class_weight={
                    0: 1.0,
                    1: balanced_positive_weight(y_train),
                },
                verbose=0,
                shuffle=True,
            )
        return model.predict(X_predict, batch_size=256, verbose=0).ravel()

    raise ValueError(f'Unknown model: {model_name}')


def select_mlp_epochs(
    train_frame,
    params,
    seed,
    sampling='none',
    n_splits=3,
):
    values = []
    labels = train_frame[TARGET].to_numpy()
    groups = train_frame['site'].to_numpy()
    splitter = GroupKFold(n_splits=n_splits)

    for fold_number, (fit_idx, valid_idx) in enumerate(
        splitter.split(train_frame, labels, groups)
    ):
        fit_frame = apply_train_sampling(
            train_frame.iloc[fit_idx].reset_index(drop=True),
            sampling,
            seed + fold_number,
        )
        valid_frame = train_frame.iloc[valid_idx].reset_index(drop=True)
        X_fit, y_fit = prepare_dense(fit_frame)
        X_valid, y_valid = prepare_dense(valid_frame)
        preprocessor = make_preprocessor_compat(scale_numeric=True)
        X_fit = preprocessor.fit_transform(X_fit).astype('float32')
        X_valid = preprocessor.transform(X_valid).astype('float32')
        _, best_epoch = fit_mlp(
            X_fit,
            y_fit,
            X_valid,
            y_valid,
            params,
            seed + fold_number,
        )
        values.append(best_epoch)

    return max(1, int(np.median(values))), values


def tune_model_on_training(
    train_frame,
    model_name,
    seed,
    sampling='none',
    n_splits=3,
    n_trials=N_TRIALS,
):
    labels = train_frame[TARGET].to_numpy()
    groups = train_frame['site'].to_numpy()
    splits = list(GroupKFold(n_splits=n_splits).split(
        train_frame,
        labels,
        groups,
    ))

    def objective(trial):
        params = suggest_params(trial, model_name)
        fold_auc = []

        for fold_number, (fit_idx, valid_idx) in enumerate(splits):
            fit_frame = train_frame.iloc[fit_idx].reset_index(drop=True)
            valid_frame = train_frame.iloc[valid_idx].reset_index(drop=True)
            probability = fit_predict_base(
                model_name,
                fit_frame,
                valid_frame,
                params,
                seed + fold_number,
                sampling=sampling,
                validation_frame=valid_frame,
            )
            auc = roc_auc_score(valid_frame[TARGET], probability)
            fold_auc.append(float(auc))
            trial.report(float(np.mean(fold_auc)), step=fold_number)
            if trial.should_prune():
                raise optuna.TrialPruned()

        return float(np.mean(fold_auc))

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=seed),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=3),
    )
    started = time.perf_counter()
    study.optimize(
        objective,
        n_trials=n_trials,
        n_jobs=1,
        show_progress_bar=False,
    )
    return study, time.perf_counter() - started

## Inner development tuning and OOF comparison

Optuna chỉ nhìn thấy các inner validation hospitals của development. Sau khi Optuna chọn params, notebook tạo lại OOF theo hospital để tính:

- pooled development ROC-AUC/PR-AUC/Recall/Brier;
- per-site metrics;
- worst-site ROC-AUC, worst-site Recall và worst-site Brier.

Các metrics này dùng để chọn hướng phát triển; Hungarian chưa được dùng.

In [ ]:
def evaluate_inner_oof(
    train_frame,
    model_name,
    params,
    sampling,
    seed,
):
    labels = train_frame[TARGET].to_numpy()
    groups = train_frame['site'].to_numpy()
    splitter = GroupKFold(n_splits=3)
    oof = np.full(len(train_frame), np.nan, dtype='float64')
    site_records = []

    for fold_number, (fit_idx, valid_idx) in enumerate(
        splitter.split(train_frame, labels, groups),
        start=1,
    ):
        fit_frame = train_frame.iloc[fit_idx].reset_index(drop=True)
        valid_frame = train_frame.iloc[valid_idx].reset_index(drop=True)
        validation_site = valid_frame['site'].iloc[0]
        started = time.perf_counter()
        probability = fit_predict_base(
            model_name,
            fit_frame,
            valid_frame,
            params,
            seed + fold_number,
            sampling=sampling,
            validation_frame=valid_frame,
        )
        fit_seconds = time.perf_counter() - started
        oof[valid_idx] = probability
        site_records.append({
            'fold': fold_number,
            'validation_site': validation_site,
            'model': model_name,
            'seed': seed,
            'fit_seconds': fit_seconds,
            **score_probability(
                valid_frame[TARGET].to_numpy(),
                probability,
            ),
        })

    assert not np.isnan(oof).any()
    pooled = {
        'model': model_name,
        'seed': seed,
        **score_probability(labels, oof),
    }
    return pd.DataFrame(site_records), pd.DataFrame([pooled])


def register_configuration(
    label,
    model_name,
    params,
    sampling,
    inner_site_records,
    inner_pooled_records,
    chosen_configurations,
):
    params = dict(params)
    fit_epoch_values = None
    if model_name == 'MLP':
        params['fit_epochs'], fit_epoch_values = select_mlp_epochs(
            development,
            params,
            FINAL_SEED + len(chosen_configurations) + 100,
            sampling=sampling,
            n_splits=3,
        )

    for seed in INNER_SEEDS:
        site_metrics, pooled_metrics = evaluate_inner_oof(
            development,
            model_name,
            params,
            sampling,
            seed,
        )
        site_metrics['configuration'] = label
        site_metrics['sampling'] = sampling
        pooled_metrics['configuration'] = label
        pooled_metrics['sampling'] = sampling
        inner_site_records.extend(site_metrics.to_dict('records'))
        inner_pooled_records.extend(pooled_metrics.to_dict('records'))

    chosen_configurations[(label, model_name)] = {
        'model': model_name,
        'params': params,
        'sampling': sampling,
        'fit_epoch_values': fit_epoch_values,
    }


chosen_configurations = {}
inner_site_records = []
inner_pooled_records = []
trial_records = []
tuning_records = []

for model_name in MODEL_NAMES:
    if RUN_BASELINE and model_name in BASELINE_MODEL_NAMES:
        register_configuration(
            'F0_P1_fixed',
            model_name,
            FIXED_PARAMS[model_name],
            'none',
            inner_site_records,
            inner_pooled_records,
            chosen_configurations,
        )

    if RUN_OPTUNA:
        study, tuning_seconds = tune_model_on_training(
            development,
            model_name,
            FINAL_SEED + MODEL_NAMES.index(model_name),
            sampling='none',
            n_splits=3,
            n_trials=N_TRIALS,
        )
        tuned_params = {**FIXED_PARAMS[model_name], **study.best_trial.params}
        tuning_records.append({
            'configuration': 'F1_P1_optuna_nested',
            'model': model_name,
            'best_inner_roc_auc': study.best_value,
            'tuning_seconds': tuning_seconds,
            'n_trials': len(study.trials),
            'best_params': json.dumps(tuned_params, sort_keys=True),
        })
        for trial in study.trials:
            trial_records.append({
                'configuration': 'F1_P1_optuna_nested',
                'model': model_name,
                'trial_number': trial.number,
                'state': str(trial.state),
                'value': trial.value,
                'params': json.dumps(trial.params, sort_keys=True),
            })
        register_configuration(
            'F1_P1_optuna_nested',
            model_name,
            tuned_params,
            'none',
            inner_site_records,
            inner_pooled_records,
            chosen_configurations,
        )

    if RUN_SMOTENC and model_name != 'CatBoost':
        study, tuning_seconds = tune_model_on_training(
            development,
            model_name,
            FINAL_SEED + MODEL_NAMES.index(model_name) + 20,
            sampling='smotenc',
            n_splits=3,
            n_trials=N_TRIALS,
        )
        smote_params = {**FIXED_PARAMS[model_name], **study.best_trial.params}
        tuning_records.append({
            'configuration': 'F2_P1_smotenc_optuna_nested',
            'model': model_name,
            'best_inner_roc_auc': study.best_value,
            'tuning_seconds': tuning_seconds,
            'n_trials': len(study.trials),
            'best_params': json.dumps(smote_params, sort_keys=True),
        })
        for trial in study.trials:
            trial_records.append({
                'configuration': 'F2_P1_smotenc_optuna_nested',
                'model': model_name,
                'trial_number': trial.number,
                'state': str(trial.state),
                'value': trial.value,
                'params': json.dumps(trial.params, sort_keys=True),
            })
        register_configuration(
            'F2_P1_smotenc_optuna_nested',
            model_name,
            smote_params,
            'smotenc',
            inner_site_records,
            inner_pooled_records,
            chosen_configurations,
        )

    if RUN_SYNTHETIC and model_name in BASELINE_MODEL_NAMES:
        register_configuration(
            'F3_P1_bootstrap_jitter_fixed',
            model_name,
            FIXED_PARAMS[model_name],
            'bootstrap_jitter',
            inner_site_records,
            inner_pooled_records,
            chosen_configurations,
        )

inner_site_metrics = pd.DataFrame(inner_site_records)
inner_pooled_metrics = pd.DataFrame(inner_pooled_records)
tuning_df = pd.DataFrame(tuning_records)
trials_df = pd.DataFrame(trial_records)

print('Configurations evaluated:', sorted(chosen_configurations))
display(tuning_df.round(4))

In [ ]:
inner_site_summary = inner_site_metrics.groupby(
    ['configuration', 'model', 'validation_site']
).agg(
    roc_auc=('roc_auc', 'mean'),
    pr_auc=('pr_auc', 'mean'),
    recall=('recall', 'mean'),
    specificity=('specificity', 'mean'),
    f1=('f1', 'mean'),
    brier=('brier', 'mean'),
).reset_index()

inner_worst_site = inner_site_summary.groupby(
    ['configuration', 'model']
).agg(
    roc_auc_worst=('roc_auc', 'min'),
    pr_auc_worst=('pr_auc', 'min'),
    recall_worst=('recall', 'min'),
    brier_worst=('brier', 'max'),
).reset_index()

inner_summary = inner_pooled_metrics.groupby(
    ['configuration', 'model']
).agg(
    roc_auc_mean=('roc_auc', 'mean'),
    roc_auc_seed_std=('roc_auc', 'std'),
    pr_auc_mean=('pr_auc', 'mean'),
    recall_mean=('recall', 'mean'),
    recall_seed_std=('recall', 'std'),
    specificity_mean=('specificity', 'mean'),
    f1_mean=('f1', 'mean'),
    brier_mean=('brier', 'mean'),
    false_negatives_mean=('false_negatives', 'mean'),
).reset_index().merge(
    inner_worst_site,
    on=['configuration', 'model'],
)

print('DEVELOPMENT-ONLY RANKING')
display(inner_summary.sort_values(
    ['roc_auc_worst', 'recall_worst', 'brier_mean'],
    ascending=[False, False, True],
).round(6))

print('DEVELOPMENT SITE METRICS')
display(inner_site_summary.sort_values(
    ['configuration', 'model', 'validation_site']
).round(6))

fig, axes = plt.subplots(1, 2, figsize=(17, 5))
sns.barplot(
    data=inner_summary,
    x='model',
    y='roc_auc_worst',
    hue='configuration',
    ax=axes[0],
)
axes[0].set_title('Worst-site inner ROC-AUC')
axes[0].tick_params(axis='x', rotation=20)
axes[0].set_ylim(0.4, 1.0)

sns.barplot(
    data=inner_summary,
    x='model',
    y='recall_worst',
    hue='configuration',
    ax=axes[1],
)
axes[1].set_title('Worst-site inner Recall @ 0.50')
axes[1].tick_params(axis='x', rotation=20)
axes[1].set_ylim(0.0, 1.0)

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / 'development_advanced_comparison.png',
    dpi=180,
    bbox_inches='tight',
)
plt.show()

## Nested OOF stacking gate

Stacking không được fit trên test. Mỗi OOF hospital được dự đoán bởi các base models đã được tune trên hai hospital còn lại; Logistic Regression meta-model chỉ học trên các OOF predictions.

Stacking chỉ được bật nếu:

    worst-site OOF ROC-AUC tăng >= 0.005
    và
    worst-site OOF Recall tăng >= 0.005

Nếu gate không đạt, notebook không tạo stacked final prediction.

In [ ]:
def make_nested_stacking_oof(train_frame, seed):
    labels = train_frame[TARGET].reset_index(drop=True).to_numpy()
    groups = train_frame['site'].reset_index(drop=True).to_numpy()
    oof = pd.DataFrame(
        index=np.arange(len(train_frame)),
        columns=MODEL_NAMES,
        dtype='float64',
    )

    outer_splits = list(GroupKFold(n_splits=3).split(
        train_frame,
        labels,
        groups,
    ))
    nested_tuning = []

    for outer_fold, (fit_idx, valid_idx) in enumerate(
        outer_splits,
        start=1,
    ):
        fit_frame = train_frame.iloc[fit_idx].reset_index(drop=True)
        valid_frame = train_frame.iloc[valid_idx].reset_index(drop=True)
        validation_site = valid_frame['site'].iloc[0]

        for model_number, model_name in enumerate(MODEL_NAMES):
            model_seed = seed + outer_fold * 100 + model_number
            study, tuning_seconds = tune_model_on_training(
                fit_frame,
                model_name,
                model_seed,
                sampling='none',
                n_splits=2,
                n_trials=STACKING_OOF_TRIALS,
            )
            params = {**FIXED_PARAMS[model_name], **study.best_trial.params}
            if model_name == 'MLP':
                params['fit_epochs'], _ = select_mlp_epochs(
                    fit_frame,
                    params,
                    model_seed + 50,
                    sampling='none',
                    n_splits=2,
                )

            probability = fit_predict_base(
                model_name,
                fit_frame,
                valid_frame,
                params,
                model_seed,
                sampling='none',
                validation_frame=valid_frame,
            )
            oof.loc[valid_idx, model_name] = probability
            nested_tuning.append({
                'outer_fold': outer_fold,
                'validation_site': validation_site,
                'model': model_name,
                'best_inner_roc_auc': study.best_value,
                'tuning_seconds': tuning_seconds,
                'n_trials': len(study.trials),
                'best_params': json.dumps(params, sort_keys=True),
            })

    assert not oof.isna().any().any()
    meta_model = LogisticRegression(
        C=1.0,
        max_iter=2000,
        class_weight='balanced',
    )
    meta_model.fit(oof[MODEL_NAMES], labels)
    oof['Stacking'] = meta_model.predict_proba(oof[MODEL_NAMES])[:, 1]

    rows = []
    site_labels = train_frame['site'].reset_index(drop=True)
    for model_name in MODEL_NAMES + ['Stacking']:
        for site in sorted(site_labels.unique()):
            mask = site_labels == site
            rows.append({
                'model': model_name,
                'site': site,
                **score_probability(
                    labels[mask],
                    oof.loc[mask, model_name].to_numpy(),
                ),
            })

    site_metrics = pd.DataFrame(rows)
    worst_metrics = site_metrics.groupby('model').agg(
        roc_auc_worst=('roc_auc', 'min'),
        recall_worst=('recall', 'min'),
        pr_auc_worst=('pr_auc', 'min'),
        brier_worst=('brier', 'max'),
    ).reset_index()
    base_reference = worst_metrics[
        worst_metrics['model'].isin(MODEL_NAMES)
    ].sort_values(
        ['roc_auc_worst', 'recall_worst'],
        ascending=False,
    ).iloc[0]
    stack_reference = worst_metrics[
        worst_metrics['model'] == 'Stacking'
    ].iloc[0]
    enabled = (
        stack_reference['roc_auc_worst']
        >= base_reference['roc_auc_worst'] + STACKING_MIN_IMPROVEMENT
        and stack_reference['recall_worst']
        >= base_reference['recall_worst'] + STACKING_MIN_IMPROVEMENT
    )
    gate = {
        'enabled': bool(enabled),
        'reference_model': base_reference['model'],
        'reference_oof_worst_auc': float(base_reference['roc_auc_worst']),
        'reference_oof_worst_recall': float(base_reference['recall_worst']),
        'stacking_oof_worst_auc': float(stack_reference['roc_auc_worst']),
        'stacking_oof_worst_recall': float(stack_reference['recall_worst']),
        'min_required_improvement': STACKING_MIN_IMPROVEMENT,
    }
    return oof, meta_model, site_metrics, pd.DataFrame(nested_tuning), gate


stacking_gate = {'enabled': False, 'reason': 'disabled by flag'}
stacking_oof = None
stacking_meta_model = None
stacking_oof_site_metrics = pd.DataFrame()
stacking_nested_tuning = pd.DataFrame()

if RUN_STACKING:
    (
        stacking_oof,
        stacking_meta_model,
        stacking_oof_site_metrics,
        stacking_nested_tuning,
        stacking_gate,
    ) = make_nested_stacking_oof(
        development,
        FINAL_SEED + 500,
    )
    print('Stacking gate:')
    display(pd.DataFrame([stacking_gate]))
    print('Stacking OOF site metrics:')
    display(stacking_oof_site_metrics.round(6))
else:
    print('Stacking disabled by flag.')

## Final fit and fixed Hungarian test

Chỉ chạy cell này sau khi đã khóa:

- danh sách configurations;
- preprocessing;
- threshold 0.50;
- final seed 42;
- stacking gate.

Mỗi configuration/model được fit trên toàn bộ Cleveland + Switzerland + VA và đánh giá Hungarian một lần. Không có tuning hoặc model selection sau cell này.

In [ ]:
# FINAL TEST CELL — run once after locking all decisions.
final_test_records = []
final_test_predictions = []
final_probabilities = {}

for (label, model_name), specification in chosen_configurations.items():
    model_name = specification['model']
    params = specification['params']
    sampling = specification['sampling']

    pipeline_started = time.perf_counter()
    probability = fit_predict_base(
        model_name,
        development,
        final_test,
        params,
        FINAL_SEED,
        sampling=sampling,
        validation_frame=None,
    )
    fit_seconds = time.perf_counter() - pipeline_started
    final_probabilities[(label, model_name)] = probability

    final_test_records.append({
        'evaluation': 'fixed_hungarian_final_test',
        'configuration': label,
        'model': model_name,
        'test_site': FINAL_TEST_SITE,
        'seed': FINAL_SEED,
        'fit_seconds': fit_seconds,
        **score_probability(
            final_test[TARGET].to_numpy(),
            probability,
        ),
    })
    final_test_predictions.append(pd.DataFrame({
        'row_index_in_hungarian': np.arange(len(final_test)),
        'configuration': label,
        'model': model_name,
        'y_true': final_test[TARGET].to_numpy(),
        'probability': probability,
        'prediction_at_0_50': (probability >= THRESHOLD).astype('int8'),
    }))

if (
    RUN_STACKING
    and stacking_gate.get('enabled', False)
    and stacking_meta_model is not None
    and all(
        ('F1_P1_optuna_nested', model_name) in final_probabilities
        for model_name in MODEL_NAMES
    )
):
    stacked_features = np.column_stack([
        final_probabilities[('F1_P1_optuna_nested', model_name)]
        for model_name in MODEL_NAMES
    ])
    stacking_probability = stacking_meta_model.predict_proba(
        stacked_features
    )[:, 1]
    final_test_records.append({
        'evaluation': 'fixed_hungarian_final_test',
        'configuration': 'stacking_oof_gated',
        'model': 'Stacking',
        'test_site': FINAL_TEST_SITE,
        'seed': FINAL_SEED,
        'fit_seconds': np.nan,
        **score_probability(
            final_test[TARGET].to_numpy(),
            stacking_probability,
        ),
    })
    final_test_predictions.append(pd.DataFrame({
        'row_index_in_hungarian': np.arange(len(final_test)),
        'configuration': 'stacking_oof_gated',
        'model': 'Stacking',
        'y_true': final_test[TARGET].to_numpy(),
        'probability': stacking_probability,
        'prediction_at_0_50': (
            stacking_probability >= THRESHOLD
        ).astype('int8'),
    }))

final_test_results = pd.DataFrame(final_test_records)
final_test_predictions = pd.concat(
    final_test_predictions,
    ignore_index=True,
)

print('FINAL FIXED HUNGARIAN TEST')
display(final_test_results.sort_values(
    ['roc_auc', 'recall'],
    ascending=False,
).round(6))
print('Do not retune after viewing this table.')

In [ ]:
inner_site_metrics.to_csv(
    OUTPUT_DIR / 'inner_site_metrics.csv',
    index=False,
)
inner_site_summary.to_csv(
    OUTPUT_DIR / 'inner_site_summary.csv',
    index=False,
)
inner_pooled_metrics.to_csv(
    OUTPUT_DIR / 'inner_pooled_metrics.csv',
    index=False,
)
inner_summary.to_csv(
    OUTPUT_DIR / 'inner_summary.csv',
    index=False,
)
tuning_df.to_csv(
    OUTPUT_DIR / 'optuna_tuning_summary.csv',
    index=False,
)
trials_df.to_csv(
    OUTPUT_DIR / 'optuna_trial_history.csv',
    index=False,
)
final_test_results.to_csv(
    OUTPUT_DIR / 'final_hungarian_test_results.csv',
    index=False,
)
final_test_predictions.to_csv(
    OUTPUT_DIR / 'final_hungarian_predictions.csv',
    index=False,
)

if not stacking_oof_site_metrics.empty:
    stacking_oof.to_csv(
        OUTPUT_DIR / 'stacking_development_oof_predictions.csv',
        index=False,
    )
    stacking_oof_site_metrics.to_csv(
        OUTPUT_DIR / 'stacking_oof_site_metrics.csv',
        index=False,
    )
    stacking_nested_tuning.to_csv(
        OUTPUT_DIR / 'stacking_nested_tuning.csv',
        index=False,
    )
pd.DataFrame([stacking_gate]).to_csv(
    OUTPUT_DIR / 'stacking_gate.csv',
    index=False,
)

run_config = {
    'notebook': '27_UCI_Multicenter_Fixed_Hungarian_Advanced_Colab',
    'development_sites': DEVELOPMENT_SITES,
    'final_test_site': FINAL_TEST_SITE,
    'validation': (
        'Inner 3-fold GroupKFold by hospital; '
        '2 train hospitals + 1 validation hospital'
    ),
    'final_fit': 'Cleveland + Switzerland + VA',
    'models': MODEL_NAMES,
    'baseline_models': BASELINE_MODEL_NAMES,
    'n_trials': N_TRIALS,
    'stacking_oof_trials': STACKING_OOF_TRIALS,
    'threshold': THRESHOLD,
    'inner_seeds': list(INNER_SEEDS),
    'final_seed': FINAL_SEED,
    'flags': {
        'run_baseline': RUN_BASELINE,
        'run_optuna': RUN_OPTUNA,
        'run_smotenc': RUN_SMOTENC,
        'run_synthetic': RUN_SYNTHETIC,
        'run_stacking': RUN_STACKING,
    },
    'synthetic_method': (
        'train-only bootstrap jitter; disabled by default and requires '
        'medical plausibility review'
    ),
    'stacking_gate': (
        'nested development OOF worst-site ROC-AUC and Recall each improve '
        f'by at least {STACKING_MIN_IMPROVEMENT}'
    ),
    'test_policy': (
        'Hungarian excluded from development preprocessing, tuning, '
        'threshold selection and stacking gate; evaluated in final cell'
    ),
    'exploratory_caveat': (
        'LOCO results from Colab 24/25 were already inspected, so this is '
        'not a blind absolute external evaluation'
    ),
    'data_source': str(LOCAL_DATA_DIR) if LOCAL_DATA_DIR else 'UCI URL fallback',
}
(OUTPUT_DIR / 'run_config.json').write_text(
    json.dumps(run_config, indent=2),
    encoding='utf-8',
)

zip_path = shutil.make_archive(
    '/content/uci_multicenter_fixed_hungarian_advanced_results',
    'zip',
    OUTPUT_DIR,
)
print('Saved artifacts:', OUTPUT_DIR)
print('ZIP:', zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Not running in Colab; ZIP remains at:', zip_path)

## Colab 27 checklist

- [x] Hungarian cố định làm external holdout.
- [x] Cleveland, Switzerland, VA chỉ dùng cho development.
- [x] Inner CV theo hospital: 2 train + 1 validation.
- [x] Final fit trên cả 3 development hospitals.
- [x] P1 và preprocessing fit trong từng training fold.
- [x] Baseline cũ F0: LR + LightGBM, class-balanced, threshold 0.50.
- [x] Optuna nested chỉ trên development.
- [x] SMOTENC chỉ tạo mẫu từ training split.
- [x] Synthetic augmentation có nhánh train-only, tắt mặc định để tránh giả định y khoa không có căn cứ.
- [x] Stacking nested OOF và có gate worst-site AUC/Recall.
- [x] Hungarian chỉ được đánh giá ở cell final test.
- [ ] Không điều chỉnh model sau khi xem Hungarian.
- [ ] Muốn blind tuyệt đối cần hospital/dataset thứ 5.